In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import urllib.parse

from sklearn.metrics.pairwise import cosine_similarity

### Loading Animelist

In [2]:
df_AnimeList = pd.DataFrame()
for f in list(Path('../Scraping/Season').rglob('*.csv')):
    if re.search(r'.ipynb_checkpoints', str(f)) is not None:
        continue

    df_Read = pd.read_csv(f)
    df_AnimeList = pd.concat([df_AnimeList, df_Read])

del df_Read

In [3]:
df_AnimeList['MAL_id'] = df_AnimeList['Link'].apply(lambda x : int(re.findall(r'anime/(\d+)',x)[0]))
df_AnimeList = df_AnimeList.drop(columns=['Link', 'Unnamed: 0'])
df_AnimeList = df_AnimeList.set_index('MAL_id')
df_AnimeList = df_AnimeList.drop_duplicates()
df_AnimeList

,Title
MAL_id,
30,Shinseiki Evangelion
852,Gokinjo Monogatari
495,Bakuretsu Hunters
1567,Kaitou Saint Tail
116,Shinpi no Sekai El-Hazard (TV)
...,...
54145,Cardfight!! Vanguard: D Series Season 9
53722,Bullet/Bullet
60726,My Melody & Kuromi


### Loading User Ratings

In [4]:
df_Raw = pd.DataFrame()
for f in list(Path('../Scraping/Users/Watchlist').rglob('*.csv')):
    if re.search(r'.ipynb_checkpoints', str(f)) is not None:
        continue

    df_Read = pd.read_csv(f)
    df_Read['User'] = f.name.replace('.csv', '')
    df_Raw = pd.concat([df_Raw, df_Read])

del df_Read

In [5]:
df_Raw = df_Raw.drop(columns=['Unnamed: 0'])
df_Raw = df_Raw.dropna()

df_Raw['MAL_id'] = df_Raw['MAL_id'].astype(int)
df_Raw['Title'] = df_Raw['Title'].apply(lambda x : urllib.parse.unquote(x).replace('_', ' '))
df_Raw

,MAL_id,Title,Score,Type,User
2,1735,Naruto Shippuuden,9.0,TV,-Ari_Su-
5,8481,Bungaku Shoujo Memoire,8.0,OVA,-Ari_Su-
6,5525,07-Ghost,10.0,TV,-Ari_Su-
7,1689,Byousoku 5 Centimeter,9.0,Movie,-Ari_Su-
8,3210,Ikoku Irokoi Romantan,8.0,OVA,-Ari_Su-
...,...,...,...,...,...
15,19815,No Game No Life,10.0,TV,____hentxi____
16,24991,No Game No Life Specials,10.0,Special,____hentxi____
17,33674,No Game No Life Zero,9.0,Movie,____hentxi____
18,37402,No Game No Life Zero - Manner Movie,10.0,Movie,____hentxi____


### User-Anime Score Matrix

In [6]:
matrix_Raw = df_Raw.pivot_table(index='User', columns='MAL_id', values='Score', aggfunc='mean')
matrix_Raw = matrix_Raw.fillna(0)
matrix_Raw = matrix_Raw.astype(np.int8)
matrix_Raw

MAL_id,1,5,6,7,8,15,16,17,18,19,...,60629,60662,60666,60725,60736,60749,60772,60811,61130,61134
User,,,,,,,,,,,,,,,,,,,,,
-Ari_Su-,0,0,0,5,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
-Ellisson-,8,0,0,0,0,0,0,0,0,10,...,0,0,0,0,0,0,0,0,0,0
-HonokaKousaka-,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
-Lofn-,8,0,5,0,0,0,0,0,0,7,...,0,0,0,0,0,0,0,0,0,0
-Mayu,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
zilch16,5,0,0,7,0,9,0,0,9,0,...,0,0,0,0,0,0,0,0,0,0
zoooper,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
zororonoa,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### User-User Similarity Matrix

In [7]:
matrix_Users = pd.DataFrame(cosine_similarity(matrix_Raw), index=matrix_Raw.index, columns=matrix_Raw.index)
matrix_Users.head(10)

User,-Ari_Su-,-Ellisson-,-HonokaKousaka-,-Lofn-,-Mayu,-Norza-,-PJ-,-R2J-,-Trinody-,-Xythus-,...,zenjamibu,zerevyc,zhongguoshidiyi,zhorahh,zikangue,zilch16,zoooper,zororonoa,zwrei,zzombgrl
User,,,,,,,,,,,,,,,,,,,,,
-Ari_Su-,1.000000,0.179281,0.185241,0.271223,0.240850,0.090317,0.121379,0.242395,0.230445,0.239614,...,0.050116,0.077523,0.000000,0.064538,0.184481,0.152889,0.083652,0.161892,0.066484,0.128767
-Ellisson-,0.179281,1.000000,0.318937,0.260775,0.225429,0.262912,0.191138,0.303642,0.222723,0.371772,...,0.055872,0.078853,0.000000,0.125442,0.330626,0.069273,0.228419,0.237212,0.103906,0.136185
-HonokaKousaka-,0.185241,0.318937,1.000000,0.223382,0.210315,0.198153,0.208382,0.238033,0.238584,0.349097,...,0.111218,0.037055,0.000000,0.080724,0.200075,0.060039,0.114037,0.142456,0.052585,0.097299
-Lofn-,0.271223,0.260775,0.223382,1.000000,0.193777,0.175935,0.137197,0.317377,0.289033,0.277247,...,0.082978,0.078260,0.029348,0.110815,0.253140,0.185721,0.133960,0.153041,0.013882,0.098812
-Mayu,0.240850,0.225429,0.210315,0.193777,1.000000,0.098186,0.204495,0.209410,0.231688,0.265812,...,0.026473,0.072340,0.000000,0.105865,0.107654,0.098696,0.025400,0.038647,0.000000,0.114676
-Norza-,0.090317,0.262912,0.198153,0.175935,0.098186,1.000000,0.167685,0.170231,0.305684,0.248460,...,0.097872,0.104006,0.091770,0.000000,0.254551,0.036748,0.103112,0.109167,0.082479,0.017506
-PJ-,0.121379,0.191138,0.208382,0.137197,0.204495,0.167685,1.000000,0.156941,0.192565,0.240748,...,0.040366,0.000000,0.000000,0.080599,0.144389,0.091448,0.061855,0.107196,0.000000,0.015444
-R2J-,0.242395,0.303642,0.238033,0.317377,0.209410,0.170231,0.156941,1.000000,0.203283,0.278408,...,0.115940,0.065527,0.027114,0.064688,0.283746,0.164609,0.241980,0.218169,0.032065,0.109978
-Trinody-,0.230445,0.222723,0.238584,0.289033,0.231688,0.305684,0.192565,0.203283,1.000000,0.379080,...,0.080210,0.096709,0.070571,0.049068,0.274617,0.089413,0.151429,0.173430,0.000000,0.042032


## Test User Actual_Hana
Gather Top 10 Most Similar Users

In [8]:
USER = 'Actual_Hana'

similar_users = matrix_Users.loc[USER].sort_values(ascending=False)[1:11]
similar_users

User
_Mei_           0.407530
jasmine_kb08    0.392706
xiuzuki         0.385509
MysticMira      0.374602
Akachin69       0.373102
Satoshiwa       0.371323
rururaruri      0.370735
gellai          0.365401
DogAlchemist    0.363518
lunatricks      0.362840
Name: Actual_Hana, dtype: float64

## Shows User Watched. Others have Watched Some

In [9]:
alreadyWatched = matrix_Raw.loc[USER] > 0
matrix_Raw.loc[np.append(USER, similar_users.index), alreadyWatched]

MAL_id,20,22,121,123,226,270,355,356,384,853,...,22145,22199,22319,22535,23273,23283,23317,23673,27899,28223
User,,,,,,,,,,,,,,,,,,,,,
Actual_Hana,10,9,10,10,9,9,9,9,10,9,...,8,9,10,10,10,10,8,10,10,10
_Mei_,0,0,9,0,0,9,0,7,0,9,...,0,7,9,8,9,8,0,0,6,0
jasmine_kb08,0,0,0,7,7,0,7,6,0,0,...,0,0,0,0,0,0,0,0,0,0
xiuzuki,10,0,6,0,0,0,10,0,0,8,...,0,8,8,0,8,0,0,0,9,0
MysticMira,4,0,0,0,0,0,0,0,0,6,...,8,7,7,8,8,0,0,0,9,0
Akachin69,10,0,0,0,7,7,0,0,0,7,...,0,0,0,0,0,0,0,0,0,0
Satoshiwa,8,0,0,0,0,0,4,0,0,0,...,0,7,8,0,6,6,0,0,7,7
rururaruri,9,0,0,0,0,0,0,0,0,0,...,7,9,9,10,10,10,0,7,8,9
gellai,0,0,0,0,4,0,0,0,0,10,...,0,0,7,0,7,0,0,0,0,0


## Shows User hasn't Watched. Others have

In [10]:
matrix_Other_User_Ratings = matrix_Raw.loc[similar_users.index, ~alreadyWatched]
matrix_Other_User_Ratings

MAL_id,1,5,6,7,8,15,16,17,18,19,...,60629,60662,60666,60725,60736,60749,60772,60811,61130,61134
User,,,,,,,,,,,,,,,,,,,,,
_Mei_,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
jasmine_kb08,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
xiuzuki,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
MysticMira,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Akachin69,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Satoshiwa,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
rururaruri,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
gellai,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DogAlchemist,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Recommendations' Avg Score by Similar Users

In [11]:
reccomended_Shows = matrix_Other_User_Ratings.mean(axis=0).sort_values(ascending=False)[:10]
df_Reccomendations = df_AnimeList.loc[reccomended_Shows.index].copy()
df_Reccomendations['Avg_Score'] = reccomended_Shows.values
df_Reccomendations

,Title,Avg_Score
MAL_id,,
11111,Another,8.4
7054,Kaichou wa Maid-sama!,7.0
9919,Ao no Exorcist,6.7
6702,Fairy Tail,6.5
4181,Clannad: After Story,6.1
14741,Chuunibyou demo Koi ga Shitai!,5.8
9989,Ano Hi Mita Hana no Namae wo Bokutachi wa Mada...,5.7
13601,Psycho-Pass,5.0
30276,One Punch Man,5.0


In [12]:
def GetPersonalTop(USER, n = 10):
    USER_top = matrix_Raw.loc[USER].sort_values(ascending=False)[:n]
    
    if USER_top.index.isin(df_AnimeList.index).all():
        df = df_AnimeList.loc[USER_top.index].copy()
    else:
        df = df_Raw[['MAL_id', 'Title']].copy().set_index('MAL_id').loc[USER_top.index].drop_duplicates()
    
    df['Personal_Score'] = USER_top.values
    df = df.rename(columns={'Title': f"{USER}'s Top {n}"})
    return df

    
def Recommend(USER, n = 10):
    similar_users = matrix_Users.loc[USER].sort_values(ascending=False)[1:n+1]
    alreadyWatched = matrix_Raw.loc[USER] > 0
    matrix_Other_User_Ratings = matrix_Raw.loc[similar_users.index, ~alreadyWatched]

    reccomended_Shows = matrix_Other_User_Ratings.mean(axis=0).sort_values(ascending=False)[:n]

    if reccomended_Shows.index.isin(df_AnimeList.index).all():
        df = df_AnimeList.loc[reccomended_Shows.index].copy()
    else:
        df = df_Raw[['MAL_id', 'Title']].copy().set_index('MAL_id').loc[reccomended_Shows.index].drop_duplicates()
        
    df['Avg_Recommenders_Score'] = reccomended_Shows.values
    df = df.rename(columns={'Title': f"Others' Top {n}"})
        
    return df

## Test Mainstream Viewer

In [13]:
USER = '-Norza-'
pd.concat([GetPersonalTop(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,-Norza-'s Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,16498,Shingeki no Kyojin,10,50265,Spy x Family,9.1
1,39547,Yahari Ore no Seishun Love Comedy wa Machigatt...,10,37450,Seishun Buta Yarou wa Bunny Girl Senpai no Yum...,8.8
2,23847,Yahari Ore no Seishun Love Comedy wa Machigatt...,10,41389,Tonikaku Kawaii,7.9
3,578,Hotaru no Haka,10,43608,Kaguya-sama wa Kokurasetai: Ultra Romantic,7.8
4,31240,Re Zero kara Hajimeru Isekai Seikatsu,10,23273,Shigatsu wa Kimi no Uso,7.3
5,34134,One Punch Man 2nd Season,10,15809,Hataraku Maou-sama!,7.1
6,32281,Kimi no Na wa,10,48736,Sono Bisque Doll wa Koi wo Suru,7.1
7,38040,Kono Subarashii Sekai ni Shukufuku wo Movie K...,10,37475,Kishuku Gakkou no Juliet,7.0
8,40456,Kimetsu no Yaiba Movie Mugen Ressha-hen,10,40748,Jujutsu Kaisen,7.0
9,42203,Re Zero kara Hajimeru Isekai Seikatsu 2nd Seas...,10,4224,Toradora!,6.8


## Test Watched Only One Punch Man (Cold Start)

In [14]:
USER = 'EXUPUROOOSHUN'
pd.concat([GetPersonalTop(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,EXUPUROOOSHUN's Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,30276,One Punch Man,10,32281,Kimi no Na wa.,1.9
1,1,Cowboy Bebop,0,28851,Koe no Katachi,1.9
2,34925,Butten Monogatari,0,31240,Re:Zero kara Hajimeru Isekai Seikatsu,1.8
3,34899,Monmusu Quest,0,22319,Tokyo Ghoul,1.8
4,34901,Beyblade Burst God,0,18679,Kill la Kill,1.0
5,34902,Tsurezure Children,0,185,Initial D First Stage,1.0
6,34903,RoboMasters the Animated Series,0,28121,Dungeon ni Deai wo Motomeru no wa Machigatteir...,1.0
7,34904,Youjo Senki Senkyou Houkoku,0,2002,Heroic Age,1.0
8,34907,Through the Windows,0,5040,One Outs,1.0
9,34914,New Game,0,27631,God Eater,1.0


## Test Max Ranker : Everything 10

In [15]:
USER = 'Muhclis'
pd.concat([GetPersonalTop(USER,100).reset_index(), Recommend(USER,100).reset_index()], axis=1)

,MAL_id,Muhclis's Top 100,Personal_Score,MAL_id,Others' Top 100,Avg_Recommenders_Score
0,10686,Naruto: Honoo no Chuunin Shiken! Naruto vs. Ko...,10,31240,Re:Zero kara Hajimeru Isekai Seikatsu,6.10
1,51180,Youkoso Jitsuryoku Shijou Shugi no Kyoushitsu ...,10,39587,Re:Zero kara Hajimeru Isekai Seikatsu 2nd Season,5.40
2,7612,Haha Sange,10,48583,Shingeki no Kyojin: The Final Season Part 2,5.13
3,51088,Eroriman,10,33486,Boku no Hero Academia 2nd Season,4.92
4,51096,Youkoso Jitsuryoku Shijou Shugi no Kyoushitsu ...,10,32182,Mob Psycho 100,4.90
...,...,...,...,...,...,...
95,31933,JoJo no Kimyou na Bouken Part 4: Diamond wa Ku...,10,27899,Tokyo Ghoul √A,2.71
96,50307,Tonikaku Kawaii 2nd Season,10,28025,Tsukimonogatari,2.71
97,8110,Fukubiki! Triangle: Miharu After,10,34572,Black Clover,2.70
98,50336,Nudist Beach ni Shuugaku Ryokou de!! The Anima...,10,35839,Sora yori mo Tooi Basho,2.69


## AOT Fan

In [16]:
USER = 'Math082r'
pd.concat([GetPersonalTop(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,Math082r's Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,2251,Baccano!,10,2904,Code Geass: Hangyaku no Lelouch R2,8.4
1,28851,Koe no Katachi,10,1575,Code Geass: Hangyaku no Lelouch,8.2
2,59571,Shingeki no Kyojin Movie: Kanketsu-hen - The L...,9,35849,Darling in the FranXX,6.8
3,48583,Shingeki no Kyojin: The Final Season Part 2,9,38000,Kimetsu no Yaiba,6.7
4,3297,Aria the Origination,9,14345,Btooom!,6.5
5,4181,Clannad: After Story,9,11843,Danshi Koukousei no Nichijou,6.5
6,40028,Shingeki no Kyojin: The Final Season,9,15583,Date A Live,6.3
7,51535,Shingeki no Kyojin: The Final Season - Kankets...,9,19163,Date A Live II,6.2
8,50183,Blue Giant,8,9989,Ano Hi Mita Hana no Namae wo Bokutachi wa Mada...,6.1
9,5081,Bakemonogatari,8,35413,Imouto sae Ireba Ii.,6.0


## Drama/Seinnen/Character Growth (Monogatari / Garden of Sinners / Vinland S2)

In [17]:
USER = 'Shizawari'
pd.concat([GetPersonalTop(USER, 15).reset_index(), Recommend(USER, 15).reset_index()], axis=1)

,MAL_id,Shizawari's Top 15,Personal_Score,MAL_id,Others' Top 15,Avg_Recommenders_Score
0,55016,Idol,10,44511,Chainsaw Man,9.000000
1,4282,Kara no Kyoukai Movie 5 Mujun Rasen,10,45576,Mushoku Tensei: Isekai Ittara Honki Dasu Part 2,8.533333
2,11597,Nisemonogatari,10,32281,Kimi no Na wa.,8.200000
3,50416,Skip to Loafer,10,48736,Sono Bisque Doll wa Koi wo Suru,8.000000
4,33352,Violet Evergarden,10,9253,Steins;Gate,7.733333
5,37105,Grand Blue,10,38691,Dr. Stone,7.666667
6,52991,Sousou no Frieren,10,48561,Jujutsu Kaisen 0 Movie,7.600000
7,5205,Kara no Kyoukai Movie 7 Satsujin Kousatsu Go,10,1535,Death Note,7.600000
8,33050,Fate stay night Movie Heavens Feel - III Spri...,10,41457,86,7.533333
9,51479,Gunjou,10,23273,Shigatsu wa Kimi no Uso,7.400000


## Gooner

In [18]:
USER = 'omanko1'
pd.concat([GetPersonalTop(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,omanko1's Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,59119,Imaria,10,38470,Saimin Seishidou,7.6
1,50192,Akane wa Tsumare Somerareru,10,11827,Kuroinu: Kedakaki Seijo wa Hakudaku ni Somaru,7.3
2,49295,Chizuru-chan Kaihatsu Nikki,10,48698,Mako-chan Kaihatsu Nikki,6.4
3,59407,Iribitari Gal ni Ma〇ko Tsukawasete Morau Hanashi,9,36720,Youkoso! Sukebe Elf no Mori e,6.3
4,48468,Uchi no Otouto Maji de Dekain Dakedo Mi ni Konai?,9,24641,Baku Ane: Otouto Shibocchau zo! The Animation,6.2
5,39530,Ecchi na Oneechan ni Shiboraretai,9,21829,Fella Pure: Mitarashi-san Chi no Jijou The Ani...,6.2
6,32587,Shoujo Ramune,9,39042,Gaki ni Modotte Yarinaoshi!!!,6.0
7,22429,Itadaki! Seieki♥,9,29575,Mankitsu Happening,6.0
8,55850,Tsuma ni Damatte Sokubaikai ni Ikun ja Nakatta,9,33322,JK Bitch ni Shiboraretai,5.5
9,55146,Kunoichi Ryoujokuden Ajisai,9,37233,Muttsuri Dosukebe Tsuyu Gibo Shimai no Honshit...,5.4


# Review
Compared to content based, this method feels more like a black box approach.

It relies on the shrared ratings between users that will often be difficult to visually interpret considering the matrix's size. More than that, most of what is recommended is Mainstream due to majority ruling.

The only most notable distinction we could make was that a Porn viewer is correctly recommendeded explicit content from other users. 

# Item-Item Approach

In [19]:
matrix_Anime = pd.DataFrame(cosine_similarity(matrix_Raw.T), index=matrix_Raw.columns, columns=matrix_Raw.columns)
matrix_Anime.head()

MAL_id,1,5,6,7,8,15,16,17,18,19,...,60629,60662,60666,60725,60736,60749,60772,60811,61130,61134
MAL_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.558955,0.386905,0.240997,0.135340,0.226431,0.241048,0.181897,0.249274,0.414029,...,0.0,0.0,0.0,0.0,0.059463,0.0,0.052856,0.0,0.059463,0.059463
5,0.558955,1.000000,0.388967,0.185751,0.094317,0.165831,0.197267,0.189277,0.221777,0.315651,...,0.0,0.0,0.0,0.0,0.106893,0.0,0.118770,0.0,0.095016,0.095016
6,0.386905,0.388967,1.000000,0.306710,0.182908,0.256733,0.188511,0.185654,0.270443,0.269966,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000
7,0.240997,0.185751,0.306710,1.000000,0.177070,0.105195,0.199474,0.309352,0.225772,0.107800,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000
8,0.135340,0.094317,0.182908,0.177070,1.000000,0.223785,0.119618,0.259074,0.128604,0.041896,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000


## Test Harem : Rent a Girlfirend

Expecting Subsequent Seasons + Other harem Anime 

In [20]:
def ReccommendByItem(ANIME_ID : int, n = 10):
    similar_ID = matrix_Anime.loc[ANIME_ID].sort_values(ascending=False)[:n+1]
    return pd.concat([df_Raw.set_index('MAL_id').loc[similar_ID.index]['Title'].drop_duplicates(), similar_ID], axis=1)

In [21]:
ReccommendByItem(40839)

,Title,40839
MAL_id,,
40839,Kanojo Okarishimasu,1.000000
37450,Seishun Buta Yarou wa Bunny Girl Senpai no Yum...,0.595651
38101,5-toubun no Hanayome,0.569610
40591,Kaguya-sama wa Kokurasetai Tensai-tachi no Ren...,0.560411
39783,5-toubun no Hanayome ∬,0.557880
39535,Mushoku Tensei Isekai Ittara Honki Dasu,0.556802
37999,Kaguya-sama wa Kokurasetai Tensai-tachi no Re...,0.544399
48736,Sono Bisque Doll wa Koi wo Suru,0.541146
39587,Re Zero kara Hajimeru Isekai Seikatsu 2nd Season,0.539585


## Test Niche Gambling : Kakegurui
Expecting shows related to competition.

In [22]:
ReccommendByItem(34933)

,Title,34933
MAL_id,,
34933,Kakegurui,1.000000
37086,Kakegurui××,0.764502
37999,Kaguya-sama wa Kokurasetai Tensai-tachi no Re...,0.580343
35507,Youkoso Jitsuryoku Shijou Shugi no Kyoushitsu e,0.550055
35849,Darling in the FranXX,0.530836
38000,Kimetsu no Yaiba,0.528418
37450,Seishun Buta Yarou wa Bunny Girl Senpai no Yum...,0.524671
28851,Koe no Katachi,0.521507
33352,Violet Evergarden,0.519724
